#### Cell 1：导入依赖，定位项目及输入、输出路径。

In [1]:
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings
from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    validate_arrow_table,
    arrow_to_pandas,
)
from config.notebook_schema_browser import display_schema_metadata

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from IPython.display import display

strategy_dir = candidate_root / "01_project_collection/JQ_strategy"
research_dir = strategy_dir / "draft_experiments_v2"
database_path = (
    strategy_dir / "financial_futures_data/data/warehouse/financial_futures.duckdb"
)
output_path = research_dir / "im_main_continuous_1m.parquet"

print("Python：", sys.executable)
print("输入数据库：", database_path)
print("后续输出：", output_path)

Python： E:\anaconda3\envs\latitude\python.exe
输入数据库： E:\Latitude_Analytics_v2\01_project_collection\JQ_strategy\financial_futures_data\data\warehouse\financial_futures.duckdb
后续输出： E:\Latitude_Analytics_v2\01_project_collection\JQ_strategy\draft_experiments_v2\im_main_continuous_1m.parquet


#### Cell 2：展示日历契约，按权威 Schema 设置读取字段和分区。

In [4]:
display_schema_metadata([FUTURES_CONTRACT_CALENDAR_SCHEMA])

calendar_table_name = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"table_name"].decode("utf-8")
)
calendar_primary_key = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"primary_key"]
    .decode("utf-8").split(",")
)
calendar_partition_columns = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"partition_columns"]
    .decode("utf-8").split(",")
)
calendar_dir = settings.futures_lake_root / "silver" / calendar_table_name

calendar_columns = [
    "contract_code", "trading_date", "list_date", "delist_date",
    "session_number", "session_start_at", "session_end_at", "minute_count",
]
calendar_read_schema = pa.schema([
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column)
    for column in calendar_columns
])
calendar_partitioning = ds.partitioning(
    pa.schema([
        FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column)
        for column in calendar_partition_columns
    ]),
    flavor="hive",
)

英文表名,中文表名,字段数,主键,Hive 分区,Schema 版本
dim_futures_contract_calendar,期货合约 Session 日历维度表,21,"contract_code,trading_date,session_number","exchange_code,year,month",1.2.0


#### Cell 3：只读 IM 日线，保留后续选主力和计算换月价差所需字段。

In [3]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    daily_table = connection.execute("""
        SELECT contract_code, trading_date, close, volume, open_interest
        FROM futures_daily
        WHERE contract_code LIKE 'IM%.CCFX'
        ORDER BY trading_date, contract_code
    """).to_arrow_table()

daily_df = daily_table.to_pandas(types_mapper=pd.ArrowDtype)

if daily_df.empty:
    raise ValueError("本地数据库没有 IM 日线数据")

start_date = daily_df["trading_date"].min()
end_date = daily_df["trading_date"].max()

print(f"日线范围：{start_date} 至 {end_date}")
print(f"合约数：{daily_df['contract_code'].nunique()}；日线条数：{len(daily_df):,}")

display(
    daily_df.groupby("contract_code")["trading_date"]
    .agg(["min", "max", "count"])
)

日线范围：2022-07-22 至 2026-08-28
合约数：53；日线条数：3,984


,min,max,count
contract_code,,,
IM2208.CCFX,2022-07-22,2022-08-19,21
IM2209.CCFX,2022-07-22,2022-09-16,40
IM2210.CCFX,2022-08-22,2022-10-21,39
IM2211.CCFX,2022-09-19,2022-11-18,40
IM2212.CCFX,2022-07-22,2022-12-16,100
IM2301.CCFX,2022-11-21,2023-01-20,44
IM2302.CCFX,2022-12-19,2023-02-17,39
IM2303.CCFX,2022-07-22,2023-03-17,159
IM2304.CCFX,2023-02-20,2023-04-21,44


#### Cell 4：读取 IM 合约日历，保留 Session，并提取合约存续期和交易日序列。

In [5]:
calendar_dataset = ds.dataset(
    calendar_dir,
    format="parquet",
    partitioning=calendar_partitioning,
)
calendar_table = calendar_dataset.to_table(
    columns=calendar_columns,
    filter=(
        (ds.field("exchange_code") == "CCFX")
        & (ds.field("underlying_code") == "IM")
        & (ds.field("trading_date") >= start_date)
        & (ds.field("trading_date") <= end_date)
    ),
)
calendar_table = validate_arrow_table(calendar_table, calendar_read_schema)
calendar_df = arrow_to_pandas(calendar_table, calendar_read_schema)
calendar_df = calendar_df.sort_values(calendar_primary_key).reset_index(drop=True)

if calendar_df.empty:
    raise ValueError("正式日历中没有对应日期范围的 IM 合约记录")

contract_df = (
    calendar_df[["contract_code", "list_date", "delist_date"]]
    .drop_duplicates()
    .sort_values("contract_code")
    .reset_index(drop=True)
)
trading_dates = sorted(calendar_df["trading_date"].unique().tolist())

print(f"日历交易日数：{len(trading_dates)}；Session 条数：{len(calendar_df):,}")
display(contract_df.head())
display(calendar_df.head())

日历交易日数：996；Session 条数：7,968


,contract_code,list_date,delist_date
0,IM2208.CCFX,2022-07-22,2022-08-19
1,IM2209.CCFX,2022-07-22,2022-09-16
2,IM2210.CCFX,2022-08-22,2022-10-21
3,IM2211.CCFX,2022-09-19,2022-11-18
4,IM2212.CCFX,2022-07-22,2022-12-16


,contract_code,trading_date,list_date,delist_date,session_number,session_start_at,session_end_at,minute_count
0,IM2208.CCFX,2022-07-22,2022-07-22,2022-08-19,1,2022-07-22 09:30:00+08:00,2022-07-22 11:30:00+08:00,120
1,IM2208.CCFX,2022-07-22,2022-07-22,2022-08-19,2,2022-07-22 13:00:00+08:00,2022-07-22 15:00:00+08:00,120
2,IM2208.CCFX,2022-07-25,2022-07-22,2022-08-19,1,2022-07-25 09:30:00+08:00,2022-07-25 11:30:00+08:00,120
3,IM2208.CCFX,2022-07-25,2022-07-22,2022-08-19,2,2022-07-25 13:00:00+08:00,2022-07-25 15:00:00+08:00,120
4,IM2208.CCFX,2022-07-26,2022-07-22,2022-08-19,1,2022-07-26 09:30:00+08:00,2022-07-26 11:30:00+08:00,120


#### Cell 5：前一交易日成交量选主力。只向后月切换，门槛为严格超过 1.10 倍；
#### 现主力到期或零成交量时强制换后月。候选合约缺日线时直接报错。

In [19]:
roll_volume_ratio = 1.10
current_contract_code = None
current_delist_date = None
main_mapping_rows = []

for signal_date, trading_date in zip(trading_dates[:-1], trading_dates[1:]):
    # 信号日已上市，且执行日仍未到期的合约。
    candidate_df = contract_df.loc[
        (contract_df["list_date"] <= signal_date)
        & (contract_df["delist_date"] >= trading_date)
    ].merge(
        daily_df.loc[
            daily_df["trading_date"] == signal_date,
            ["contract_code", "volume", "open_interest"],
        ],
        on="contract_code",
        how="left",
        validate="one_to_one",
    )

    missing_contracts = candidate_df.loc[
        candidate_df["volume"].isna(), "contract_code"
    ].tolist()
    if missing_contracts:
        raise ValueError(f"{signal_date} 候选合约缺日线：{missing_contracts}")

    # 成交量并列时，依次比较持仓量、到期日和合约代码。
    candidate_df = candidate_df.loc[candidate_df["volume"] > 0].sort_values(
        ["volume", "open_interest", "delist_date", "contract_code"],
        ascending=[False, False, True, True],
    )
    if candidate_df.empty:
        raise ValueError(f"{trading_date} 没有符合条件的主力候选合约")

    if current_contract_code is None:
        selected_contract = candidate_df.iloc[0]
        selection_reason = "初始选择"
    else:
        current_candidate_df = candidate_df.loc[
            candidate_df["contract_code"] == current_contract_code
        ]
        later_candidate_df = candidate_df.loc[
            candidate_df["delist_date"] > current_delist_date
        ]

        if current_candidate_df.empty:
            if later_candidate_df.empty:
                raise ValueError(f"{trading_date} 必须换月，但没有可用后月合约")
            selected_contract = later_candidate_df.iloc[0]
            selection_reason = (
                "到期换月" if current_delist_date < trading_date else "零成交量换月"
            )
        elif (
            not later_candidate_df.empty
            and later_candidate_df.iloc[0]["volume"]
            > roll_volume_ratio * current_candidate_df.iloc[0]["volume"]
        ):
            selected_contract = later_candidate_df.iloc[0]
            selection_reason = "成交量触发换月"
        else:
            selected_contract = current_candidate_df.iloc[0]
            selection_reason = "维持主力"

    selected_contract_code = selected_contract["contract_code"]
    main_mapping_rows.append({
        "trading_date": trading_date,
        "signal_trading_date": signal_date,
        "contract_code": selected_contract_code,
        "previous_contract_code": current_contract_code,
        "is_roll_day": (
            current_contract_code is not None
            and selected_contract_code != current_contract_code
        ),
        "selected_signal_volume": float(selected_contract["volume"]),
        "selection_reason": selection_reason,
    })
    current_contract_code = selected_contract_code
    current_delist_date = selected_contract["delist_date"]

main_mapping_df = pd.DataFrame(main_mapping_rows)

if main_mapping_df.empty:
    raise ValueError("至少需要两个交易日才能生成主力映射")

main_mapping_df.head()

,trading_date,signal_trading_date,contract_code,previous_contract_code,is_roll_day,selected_signal_volume,selection_reason
0,2022-07-25,2022-07-22,IM2208.CCFX,NaN,False,22153.0,初始选择
1,2022-07-26,2022-07-25,IM2208.CCFX,IM2208.CCFX,False,22883.0,维持主力
2,2022-07-27,2022-07-26,IM2208.CCFX,IM2208.CCFX,False,19269.0,维持主力
3,2022-07-28,2022-07-27,IM2208.CCFX,IM2208.CCFX,False,19290.0,维持主力
4,2022-07-29,2022-07-28,IM2208.CCFX,IM2208.CCFX,False,22464.0,维持主力


#### Cell 6：核对信号时间，查看每日映射和全部换月记录。

In [7]:
if not (
    main_mapping_df["signal_trading_date"] < main_mapping_df["trading_date"]
).all():
    raise ValueError("主力选择信号必须早于执行日")

roll_mapping_df = main_mapping_df.loc[
    main_mapping_df["is_roll_day"]
].reset_index(drop=True)

print(f"主力覆盖：{len(main_mapping_df)} 个交易日")
print(f"换月次数：{len(roll_mapping_df)}")
print(f"首个交易日 {trading_dates[0]} 仅提供选择信号")

display(main_mapping_df.head(10))
display(roll_mapping_df)

主力覆盖：995 个交易日
换月次数：34
首个交易日 2022-07-22 仅提供选择信号


,trading_date,signal_trading_date,contract_code,previous_contract_code,is_roll_day,selected_signal_volume,selection_reason
0,2022-07-25,2022-07-22,IM2208.CCFX,NaN,False,22153.0,初始选择
1,2022-07-26,2022-07-25,IM2208.CCFX,IM2208.CCFX,False,22883.0,维持主力
2,2022-07-27,2022-07-26,IM2208.CCFX,IM2208.CCFX,False,19269.0,维持主力
3,2022-07-28,2022-07-27,IM2208.CCFX,IM2208.CCFX,False,19290.0,维持主力
4,2022-07-29,2022-07-28,IM2208.CCFX,IM2208.CCFX,False,22464.0,维持主力
5,2022-08-01,2022-07-29,IM2208.CCFX,IM2208.CCFX,False,25025.0,维持主力
6,2022-08-02,2022-08-01,IM2208.CCFX,IM2208.CCFX,False,22084.0,维持主力
7,2022-08-03,2022-08-02,IM2208.CCFX,IM2208.CCFX,False,37753.0,维持主力
8,2022-08-04,2022-08-03,IM2208.CCFX,IM2208.CCFX,False,32325.0,维持主力
9,2022-08-05,2022-08-04,IM2208.CCFX,IM2208.CCFX,False,35394.0,维持主力


,trading_date,signal_trading_date,contract_code,previous_contract_code,is_roll_day,selected_signal_volume,selection_reason
0,2022-08-19,2022-08-18,IM2209.CCFX,IM2208.CCFX,True,19946.0,成交量触发换月
1,2022-09-16,2022-09-15,IM2210.CCFX,IM2209.CCFX,True,34741.0,成交量触发换月
2,2022-10-21,2022-10-20,IM2211.CCFX,IM2210.CCFX,True,37125.0,成交量触发换月
3,2022-11-18,2022-11-17,IM2212.CCFX,IM2211.CCFX,True,31632.0,成交量触发换月
4,2022-12-19,2022-12-16,IM2301.CCFX,IM2212.CCFX,True,30237.0,到期换月
5,2023-01-20,2023-01-19,IM2302.CCFX,IM2301.CCFX,True,19184.0,成交量触发换月
6,2023-02-17,2023-02-16,IM2303.CCFX,IM2302.CCFX,True,49836.0,成交量触发换月
7,2023-03-20,2023-03-17,IM2304.CCFX,IM2303.CCFX,True,39453.0,到期换月
8,2023-04-24,2023-04-21,IM2305.CCFX,IM2304.CCFX,True,36357.0,到期换月
9,2023-05-19,2023-05-18,IM2306.CCFX,IM2305.CCFX,True,27768.0,成交量触发换月


#### Cell 7：每次换月取不晚于信号日的最近共同有效收盘价，
计算 log(新合约收盘价) − log(旧合约收盘价)。

In [8]:
import numpy as np

close_by_date_df = daily_df.pivot(
    index="trading_date",
    columns="contract_code",
    values="close",
).sort_index()

roll_gap_rows = []

for roll in roll_mapping_df.itertuples(index=False):
    common_close_df = close_by_date_df.loc[
        close_by_date_df.index <= roll.signal_trading_date,
        [roll.previous_contract_code, roll.contract_code],
    ].dropna()

    common_close_df = common_close_df.loc[
        (common_close_df > 0).all(axis=1)
    ]
    if common_close_df.empty:
        raise ValueError(
            f"{roll.trading_date} 换月缺少共同有效收盘价："
            f"{roll.previous_contract_code} → {roll.contract_code}"
        )

    reference_close = common_close_df.iloc[-1]
    old_reference_close = float(reference_close[roll.previous_contract_code])
    new_reference_close = float(reference_close[roll.contract_code])
    roll_log_gap = np.log(new_reference_close) - np.log(old_reference_close)

    roll_gap_rows.append((
        roll.trading_date,
        roll.signal_trading_date,
        reference_close.name,
        roll.previous_contract_code,
        roll.contract_code,
        old_reference_close,
        new_reference_close,
        roll_log_gap,
    ))

roll_gap_df = pd.DataFrame(
    roll_gap_rows,
    columns=[
        "trading_date", "signal_trading_date", "roll_reference_trading_date",
        "previous_contract_code", "contract_code",
        "old_reference_close", "new_reference_close", "roll_log_gap",
    ],
)

display(roll_gap_df)

,trading_date,signal_trading_date,roll_reference_trading_date,previous_contract_code,contract_code,old_reference_close,new_reference_close,roll_log_gap
0,2022-08-19,2022-08-18,2022-08-18,IM2208.CCFX,IM2209.CCFX,7358.8,7316.6,-0.005751
1,2022-09-16,2022-09-15,2022-09-15,IM2209.CCFX,IM2210.CCFX,6596.4,6518.6,-0.011864
2,2022-10-21,2022-10-20,2022-10-20,IM2210.CCFX,IM2211.CCFX,6445.8,6366.0,-0.012457
3,2022-11-18,2022-11-17,2022-11-17,IM2211.CCFX,IM2212.CCFX,6715.2,6707.0,-0.001222
4,2022-12-19,2022-12-16,2022-12-16,IM2212.CCFX,IM2301.CCFX,6515.0,6531.8,0.002575
5,2023-01-20,2023-01-19,2023-01-19,IM2301.CCFX,IM2302.CCFX,6687.4,6717.0,0.004416
6,2023-02-17,2023-02-16,2023-02-16,IM2302.CCFX,IM2303.CCFX,6929.4,6909.2,-0.002919
7,2023-03-20,2023-03-17,2023-03-17,IM2303.CCFX,IM2304.CCFX,6748.4,6726.0,-0.003325
8,2023-04-24,2023-04-21,2023-04-21,IM2304.CCFX,IM2305.CCFX,6813.4,6770.2,-0.006361
9,2023-05-19,2023-05-18,2023-05-18,IM2305.CCFX,IM2306.CCFX,6578.6,6551.8,-0.004082


#### Cell 8：把单次价差映射到换月日，再按交易日累计。非换月日的单次价差为零。

In [9]:
roll_log_gap_by_date = roll_gap_df.set_index("trading_date")["roll_log_gap"]

main_mapping_df["roll_log_gap"] = (
    main_mapping_df["trading_date"]
    .map(roll_log_gap_by_date)
    .astype("float64")
)

if (
    main_mapping_df["is_roll_day"]
    & main_mapping_df["roll_log_gap"].isna()
).any():
    raise ValueError("存在尚未计算价差的换月日，请先完成 Cell 7")

main_mapping_df["roll_log_gap"] = main_mapping_df["roll_log_gap"].fillna(0.0)
main_mapping_df["cumulative_roll_log_gap"] = (
    main_mapping_df["roll_log_gap"].cumsum()
)

print(
    "全段累计换月 log 价差："
    f"{main_mapping_df['cumulative_roll_log_gap'].iloc[-1]:.12f}"
)
display(
    main_mapping_df[
        ["trading_date", "contract_code", "is_roll_day",
         "roll_log_gap", "cumulative_roll_log_gap"]
    ].head(10)
)

全段累计换月 log 价差：-0.379603907631


,trading_date,contract_code,is_roll_day,roll_log_gap,cumulative_roll_log_gap
0,2022-07-25,IM2208.CCFX,False,0.0,0.0
1,2022-07-26,IM2208.CCFX,False,0.0,0.0
2,2022-07-27,IM2208.CCFX,False,0.0,0.0
3,2022-07-28,IM2208.CCFX,False,0.0,0.0
4,2022-07-29,IM2208.CCFX,False,0.0,0.0
5,2022-08-01,IM2208.CCFX,False,0.0,0.0
6,2022-08-02,IM2208.CCFX,False,0.0,0.0
7,2022-08-03,IM2208.CCFX,False,0.0,0.0
8,2022-08-04,IM2208.CCFX,False,0.0,0.0
9,2022-08-05,IM2208.CCFX,False,0.0,0.0


#### Cell 9：按每日主力映射读取一分钟行情，保留原始价格和累计换月价差。

In [11]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    connection.register("main_mapping", main_mapping_df)

    minute_table = connection.execute("""
        SELECT
            f.contract_code, f.trading_date, f.session_number, f.bar_at,
            f.open, f.high, f.low, f.close,
            f.volume, f.money, f.open_interest,
            m.signal_trading_date, m.is_roll_day, m.cumulative_roll_log_gap
        FROM futures_minute AS f
        JOIN main_mapping AS m
          ON f.contract_code = m.contract_code
         AND f.trading_date = m.trading_date
        ORDER BY f.bar_at
    """).to_arrow_table()

minute_df = minute_table.to_pandas(types_mapper=pd.ArrowDtype)

print(f"分钟条数：{len(minute_df):,}")
print(f"交易日数：{minute_df['trading_date'].nunique()}")
display(minute_df.head())

分钟条数：238,800
交易日数：995


,contract_code,trading_date,session_number,bar_at,open,high,low,close,volume,money,open_interest,signal_trading_date,is_roll_day,cumulative_roll_log_gap
0,IM2208.CCFX,2022-07-25,1,2022-07-25 09:31:00+08:00,6974.2,6998.0,6964.4,6973.8,664.0,927022080.0,17333.0,2022-07-22,False,0.0
1,IM2208.CCFX,2022-07-25,1,2022-07-25 09:32:00+08:00,6974.0,6995.2,6969.6,6982.8,428.0,597671840.0,17268.0,2022-07-22,False,0.0
2,IM2208.CCFX,2022-07-25,1,2022-07-25 09:33:00+08:00,6982.4,6989.8,6981.6,6989.8,280.0,391195160.0,17213.0,2022-07-22,False,0.0
3,IM2208.CCFX,2022-07-25,1,2022-07-25 09:34:00+08:00,6989.6,6993.0,6982.6,6993.0,320.0,447271480.0,17123.0,2022-07-22,False,0.0
4,IM2208.CCFX,2022-07-25,1,2022-07-25 09:35:00+08:00,6993.6,7009.0,6991.8,7002.2,286.0,400457120.0,17020.0,2022-07-22,False,0.0


#### Cell 10

In [13]:
price_columns = ["open", "high", "low", "close"]
minute_count = len(minute_df)

cumulative_roll_log_gap = minute_df[
    "cumulative_roll_log_gap"
].to_numpy(dtype=np.float64)

relative_roll_log_gap = (
    cumulative_roll_log_gap - cumulative_roll_log_gap[0]
)
total_roll_log_gap = float(relative_roll_log_gap[-1])

# 按实际交易分钟分配权重，首分钟为 0，末分钟为 1。
adjustment_weight = np.linspace(0.0, 1.0, minute_count)
log_adjustment = (
    -relative_roll_log_gap + adjustment_weight * total_roll_log_gap
)
price_multiplier = np.exp(log_adjustment)

minute_df["bidirectional_adjustment_weight"] = adjustment_weight
minute_df["bidirectional_log_adjustment"] = log_adjustment

# 复权依赖完整快照，记录其末分钟时刻。
minute_df["price_adjustment_available_at"] = minute_df["bar_at"].iloc[-1]

for price_column in price_columns:
    raw_price_values = minute_df[price_column].to_numpy(dtype=np.float64)

    minute_df[f"bidirectional_adjusted_log_{price_column}"] = (
        np.log(raw_price_values) + log_adjustment
    )
    minute_df[f"bidirectional_adjusted_{price_column}"] = (
        raw_price_values * price_multiplier
    )

In [16]:
display(minute_df.head())
display(minute_df.tail())

,contract_code,trading_date,session_number,bar_at,open,high,low,close,volume,money,open_interest,signal_trading_date,is_roll_day,cumulative_roll_log_gap,bidirectional_adjustment_weight,bidirectional_log_adjustment,price_adjustment_available_at,bidirectional_adjusted_log_open,bidirectional_adjusted_open,bidirectional_adjusted_log_high,bidirectional_adjusted_high,bidirectional_adjusted_log_low,bidirectional_adjusted_low,bidirectional_adjusted_log_close,bidirectional_adjusted_close
0,IM2208.CCFX,2022-07-25,1,2022-07-25 09:31:00+08:00,6974.2,6998.0,6964.4,6973.8,664.0,927022080.0,17333.0,2022-07-22,False,0.0,0.000000,-0.000000,2026-08-28 15:00:00+08:00,8.849973,6974.200000,8.853380,6998.000000,8.848567,6964.400000,8.849916,6973.800000
1,IM2208.CCFX,2022-07-25,1,2022-07-25 09:32:00+08:00,6974.0,6995.2,6969.6,6982.8,428.0,597671840.0,17268.0,2022-07-22,False,0.0,0.000004,-0.000002,2026-08-28 15:00:00+08:00,8.849943,6973.988914,8.852978,6995.188880,8.849312,6969.588921,8.851204,6982.788900
2,IM2208.CCFX,2022-07-25,1,2022-07-25 09:33:00+08:00,6982.4,6989.8,6981.6,6989.8,280.0,391195160.0,17213.0,2022-07-22,False,0.0,0.000008,-0.000003,2026-08-28 15:00:00+08:00,8.851145,6982.377801,8.852204,6989.777778,8.851030,6981.577804,8.852204,6989.777778
3,IM2208.CCFX,2022-07-25,1,2022-07-25 09:34:00+08:00,6989.6,6993.0,6982.6,6993.0,320.0,447271480.0,17123.0,2022-07-22,False,0.0,0.000013,-0.000005,2026-08-28 15:00:00+08:00,8.852174,6989.566667,8.852660,6992.966651,8.851172,6982.566701,8.852660,6992.966651
4,IM2208.CCFX,2022-07-25,1,2022-07-25 09:35:00+08:00,6993.6,7009.0,6991.8,7002.2,286.0,400457120.0,17020.0,2022-07-22,False,0.0,0.000017,-0.000006,2026-08-28 15:00:00+08:00,8.852744,6993.555531,8.854944,7008.955433,8.852487,6991.755542,8.853973,7002.155476


,contract_code,trading_date,session_number,bar_at,open,high,low,close,volume,money,open_interest,signal_trading_date,is_roll_day,cumulative_roll_log_gap,bidirectional_adjustment_weight,bidirectional_log_adjustment,price_adjustment_available_at,bidirectional_adjusted_log_open,bidirectional_adjusted_open,bidirectional_adjusted_log_high,bidirectional_adjusted_high,bidirectional_adjusted_log_low,bidirectional_adjusted_low,bidirectional_adjusted_log_close,bidirectional_adjusted_close
238795,IM2609.CCFX,2026-08-28,2,2026-08-28 14:56:00+08:00,7658.6,7662.4,7658.6,7661.8,910.0,1394295440.0,229845.0,2026-08-27,False,-0.379604,0.999983,0.000006,2026-08-28 15:00:00+08:00,8.943591,7658.648698,8.944087,7662.448722,8.943591,7658.648698,8.944009,7661.848718
238796,IM2609.CCFX,2026-08-28,2,2026-08-28 14:57:00+08:00,7661.6,7663.0,7660.6,7661.2,763.0,1169190720.0,229901.0,2026-08-27,False,-0.379604,0.999987,0.000005,2026-08-28 15:00:00+08:00,8.943981,7661.636538,8.944164,7663.036544,8.943850,7660.636533,8.943929,7661.236536
238797,IM2609.CCFX,2026-08-28,2,2026-08-28 14:58:00+08:00,7661.2,7666.0,7661.2,7664.8,773.0,1184940680.0,230019.0,2026-08-27,False,-0.379604,0.999992,0.000003,2026-08-28 15:00:00+08:00,8.943927,7661.224357,8.944553,7666.024372,8.943927,7661.224357,8.944397,7664.824369
238798,IM2609.CCFX,2026-08-28,2,2026-08-28 14:59:00+08:00,7664.4,7665.0,7662.6,7664.6,717.0,1099015240.0,230123.0,2026-08-27,False,-0.379604,0.999996,0.000002,2026-08-28 15:00:00+08:00,8.944343,7664.412184,8.944421,7665.012185,8.944108,7662.612181,8.944369,7664.612184
238799,IM2609.CCFX,2026-08-28,2,2026-08-28 15:00:00+08:00,7664.6,7667.4,7663.8,7666.6,1159.0,1776911160.0,230428.0,2026-08-27,False,-0.379604,1.000000,0.000000,2026-08-28 15:00:00+08:00,8.944368,7664.600000,8.944733,7667.400000,8.944263,7663.800000,8.944629,7666.600000


#### Cell 11：设置输出字段类型，保存 Parquet。

In [17]:
import pyarrow.parquet as pq

minute_schema = pa.schema([
    pa.field("contract_code", pa.string(), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False),
    pa.field("session_number", pa.int8(), nullable=False),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),

    *[
        pa.field(column, pa.float64(), nullable=False)
        for column in price_columns + ["volume", "money", "open_interest"]
    ],

    pa.field("signal_trading_date", pa.date32(), nullable=False),
    pa.field("is_roll_day", pa.bool_(), nullable=False),
    pa.field("cumulative_roll_log_gap", pa.float64(), nullable=False),
    pa.field("bidirectional_adjustment_weight", pa.float64(), nullable=False),
    pa.field("bidirectional_log_adjustment", pa.float64(), nullable=False),
    pa.field(
        "price_adjustment_available_at",
        pa.timestamp("us", tz="Asia/Shanghai"),
        nullable=False,
    ),

    *[
        pa.field(f"bidirectional_adjusted_log_{column}", pa.float64(), nullable=False)
        for column in price_columns
    ],
    *[
        pa.field(f"bidirectional_adjusted_{column}", pa.float64(), nullable=False)
        for column in price_columns
    ],
], metadata={
    b"price_adjustment_method": b"dual endpoint log adjustment",
    b"price_adjustment_formula": b"a_i=-(G_i-G_0)+i/(n-1)*(G_last-G_0)",
    b"price_adjustment_scope": b"whole input snapshot; retrospective",
    b"total_roll_log_gap": repr(total_roll_log_gap).encode("utf-8"),
})

minute_table = pa.Table.from_pandas(
    minute_df[minute_schema.names],
    schema=minute_schema,
    preserve_index=False,
)

research_dir.mkdir(parents=True, exist_ok=True)
temporary_output_path = output_path.with_suffix(".parquet.tmp")
pq.write_table(minute_table, temporary_output_path, compression="zstd")
temporary_output_path.replace(output_path)

print(f"已保存 {len(minute_df):,} 条分钟数据：{output_path}")

已保存 238,800 条分钟数据：E:\Latitude_Analytics_v2\01_project_collection\JQ_strategy\draft_experiments_v2\im_main_continuous_1m.parquet
